In [1]:
import os
import shutil
from pathlib import Path
from typing import List

import gradio as gr
import numpy as np
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None


C:\Users\msi\AppData\Local\Temp\ipykernel_7300\2201244712.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
from typing import List

stored_documents: List[Document] = []


def build_context(question: str) -> tuple[str, List[Document]]:
    if not stored_documents:
        return "No documents have been uploaded yet.", []

    texts = [doc.page_content for doc in stored_documents]
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform(texts)
    query_vec = vectorizer.transform([question])
    similarities = cosine_similarity(query_vec, matrix).ravel()
    ranked_indices = similarities.argsort()[::-1]

    top_docs = [stored_documents[i] for i in ranked_indices[:3] if similarities[i] > 0]
    if not top_docs:
        top_docs = stored_documents[:3]

    context = "\n\n".join(doc.page_content[:3000] for doc in top_docs)
    return context, top_docs


def answer_question(question: str, history):
    if not question.strip():
        return "Please enter a question."

    context, _ = build_context(question)
    if "No documents" in context:
        return context

    if openai_client:
        try:
            response = openai_client.responses.create(
                model="gpt-4.1-mini",
                input=[
                    {
                        "role": "system",
                        "content": f"You answer using only the provided context. If the answer is not in the context, say that you do not know based on the uploaded documents.\n\nContext:\n{context}",
                    },
                    {"role": "user", "content": question},
                ],
            )
            return response.output_text
        except Exception as exc:
            return f"OpenAI request failed: {exc}"

    return (
        "OpenAI API key is not configured. "
        "The app can still be used, but answers will be based on a simple similarity match from the uploaded documents.\n\n"
        f"Relevant content preview:\n{context[:2000]}"
    )


def chat_with_documents(files, question):
    if files:
        status = process_uploaded_files(files)
        if "Processed" not in status and "No readable" not in status:
            return status, None
        global stored_documents
        stored_documents = []
        for uploaded_file in files:
            if not uploaded_file:
                continue
            path = Path(uploaded_file.name)
            if not path.exists():
                continue
            text = _extract_text_from_upload(str(path))
            if text.strip():
                stored_documents.append(Document(page_content=text, metadata={"source": path.name}))

    if not question:
        return "Upload files and ask a question.", None

    answer = answer_question(question, None)
    return answer, None


OpenAI API Key exists and begins sk-proj-


In [ ]:
with gr.Blocks(title="Upload-Based RAG Chatbot") as demo:
    gr.Markdown("## Upload documents and ask questions about only their contents")
    gr.Markdown("Supported files: PDF, DOCX, TXT, PNG, JPG, JPEG")

    with gr.Row():
        files = gr.File(
            label="Upload documents",
            file_count="multiple",
            file_types=[".pdf", ".docx", ".txt", ".png", ".jpg", ".jpeg"],
        )
        chatbot = gr.Chatbot(type="messages", height=500)

    with gr.Row():
        msg = gr.Textbox(label="Question", placeholder="Ask a question about the uploaded documents")
        submit = gr.Button("Send")

    clear = gr.Button("Clear")

    def respond(files, history, question):
        if question is None:
            question = ""
        if history is None:
            history = []
        if files:
            status = process_uploaded_files(files)
            if "Processed" not in status and "No readable" not in status:
                return [("System", status)], history
            global stored_documents
            stored_documents = []
            for uploaded_file in files:
                if not uploaded_file:
                    continue
                path = Path(uploaded_file.name)
                if not path.exists():
                    continue
                text = _extract_text_from_upload(str(path))
                if text.strip():
                    stored_documents.append(Document(page_content=text, metadata={"source": path.name}))
        if not question.strip():
            return [("System", "Please enter a question.")], history
        answer = answer_question(question, history)
        history = history + [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]
        return history, history

    submit.click(respond, inputs=[files, chatbot, msg], outputs=[chatbot, chatbot])
    msg.submit(respond, inputs=[files, chatbot, msg], outputs=[chatbot, chatbot])
    clear.click(lambda: ([], []), outputs=[chatbot, chatbot])


demo.launch(share=False)
